In [1]:
# ViT-Tiny improved: unfreeze last 2 transformer blocks + custom MLP head + color jitter.
# Train on 0° only, test on 0/90/180/270.

import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

DATA_DIR = r"E:\Deep Learning Spring 26\dataset"
BATCH_SIZE = 32
EPOCHS = 6
LR = 1e-4
NUM_WORKERS = 0
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
train_dir = os.path.join(DATA_DIR, "train")
df = pd.read_csv(os.path.join(DATA_DIR, "train_labels.csv"))

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df.label, random_state=SEED)
print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")

Train: 176020  |  Val: 22002  |  Test: 22003


In [3]:
# color jitter for training (no spatial augmentation to keep rotation test clean)
train_transform = transforms.Compose([
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class CancerDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None, rotation=0):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.rotation = rotation
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row.id + ".tif")).convert("RGB")
        if self.rotation != 0:
            img = img.rotate(self.rotation)
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(row.label, dtype=torch.float32)

In [4]:
train_loader = DataLoader(CancerDataset(train_df, train_dir, train_transform, rotation=0),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(CancerDataset(val_df, train_dir, eval_transform, rotation=0),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

test_loaders = {}
for angle in [0, 90, 180, 270]:
    test_loaders[angle] = DataLoader(
        CancerDataset(test_df, train_dir, eval_transform, rotation=angle),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [5]:
# load ViT-Tiny and remove the default head
model = timm.create_model("vit_tiny_patch16_224", pretrained=True, num_classes=0, img_size=96)

# freeze everything, then selectively unfreeze last 2 blocks + norm
for param in model.parameters():
    param.requires_grad = False
for name, param in model.named_parameters():
    if "blocks.10" in name or "blocks.11" in name or "norm" in name:
        param.requires_grad = True

# custom head: 192 -> 128 -> 1
model.head = nn.Sequential(
    nn.Linear(192, 128), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(128, 1))
model = model.to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {num_params:,}")

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

Trainable params: 922,625


In [6]:
def train_one_epoch():
    model.train()
    total_loss = 0
    preds_list, labels_list = [], []
    for imgs, labels in tqdm(train_loader, desc="Training", leave=False):
        imgs = imgs.to(device)
        labels = labels.unsqueeze(1).to(device)
        out = model(imgs)
        loss = criterion(out, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds_list.extend(torch.sigmoid(out).squeeze().detach().cpu().numpy())
        labels_list.extend(labels.squeeze().cpu().numpy())
    avg_loss = total_loss / len(train_loader)
    p, l = np.array(preds_list), np.array(labels_list).astype(int)
    b = (p >= 0.5).astype(int)
    return avg_loss, accuracy_score(l, b), f1_score(l, b, average="weighted"), roc_auc_score(l, p)

def evaluate(loader):
    model.eval()
    preds_list, labels_list = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            preds_list.extend(torch.sigmoid(model(imgs)).squeeze().cpu().numpy())
            labels_list.extend(labels.numpy())
    p, l = np.array(preds_list), np.array(labels_list).astype(int)
    b = (p >= 0.5).astype(int)
    return accuracy_score(l, b), f1_score(l, b, average="weighted"), roc_auc_score(l, p)

In [7]:
best_auc = 0.0
for epoch in range(1, EPOCHS + 1):
    loss, train_acc, train_f1, train_auc = train_one_epoch()
    scheduler.step()
    val_acc, val_f1, val_auc = evaluate(val_loader)

    lr = optimizer.param_groups[0]['lr']
    print(f"\nEpoch {epoch}/{EPOCHS} (lr={lr:.2e})")
    print(f"  Train  =>  Loss: {loss:.4f}  Acc: {train_acc:.4f}  F1: {train_f1:.4f}  AUC: {train_auc:.4f}")
    print(f"  Val    =>  Acc: {val_acc:.4f}  F1: {val_f1:.4f}  AUC: {val_auc:.4f}")

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), "best_vit_improved.pth")
        print(f"  >> Saved best model (AUC={val_auc:.4f})")


--- Epoch 1/6 (lr=9.33e-05) ---
  Train  =>  Loss: 0.3552  Acc: 0.8438  F1: 0.8428  AUC: 0.9170
  Val    =>  Acc: 0.8821  F1: 0.8821  AUC: 0.9512
  >> Saved new best model (AUC=0.9512)



--- Epoch 2/6 (lr=7.50e-05) ---
  Train  =>  Loss: 0.2858  Acc: 0.8803  F1: 0.8798  AUC: 0.9465
  Val    =>  Acc: 0.8989  F1: 0.8978  AUC: 0.9627
  >> Saved new best model (AUC=0.9627)



--- Epoch 3/6 (lr=5.00e-05) ---
  Train  =>  Loss: 0.2516  Acc: 0.8964  F1: 0.8960  AUC: 0.9584
  Val    =>  Acc: 0.9121  F1: 0.9118  AUC: 0.9690
  >> Saved new best model (AUC=0.9690)



--- Epoch 4/6 (lr=2.50e-05) ---
  Train  =>  Loss: 0.2268  Acc: 0.9088  F1: 0.9084  AUC: 0.9662
  Val    =>  Acc: 0.9170  F1: 0.9169  AUC: 0.9710
  >> Saved new best model (AUC=0.9710)



--- Epoch 5/6 (lr=6.70e-06) ---
  Train  =>  Loss: 0.2050  Acc: 0.9179  F1: 0.9176  AUC: 0.9722
  Val    =>  Acc: 0.9226  F1: 0.9225  AUC: 0.9743
  >> Saved new best model (AUC=0.9743)



--- Epoch 6/6 (lr=0.00e+00) ---
  Train  =>  Loss: 0.1935  Acc: 0.9236  F1: 0.9233  AUC: 0.9752
  Val    =>  Acc: 0.9239  F1: 0.9238  AUC: 0.9758
  >> Saved new best model (AUC=0.9758)

ROTATION TEST (best model)


In [8]:
print("\nRotation test (best model):")
model.load_state_dict(torch.load("best_vit_improved.pth"))
for angle in [0, 90, 180, 270]:
    acc, f1, auc = evaluate(test_loaders[angle])
    print(f"  {angle:3d}°  =>  Acc: {acc:.4f}  F1: {f1:.4f}  AUC: {auc:.4f}")

C:\Users\AhmedFahmy\AppData\Local\Temp\ipykernel_19724\3895635646.py:265: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_vit_improved.p

    0°  =>  Acc: 0.9220  F1: 0.9220  AUC: 0.9749
   90°  =>  Acc: 0.9091  F1: 0.9089  AUC: 0.9662
  180°  =>  Acc: 0.9158  F1: 0.9157  AUC: 0.9693
  270°  =>  Acc: 0.9091  F1: 0.9089  AUC: 0.9670

Done!
